# 10 — Persistence & Deployment

Saving fitted models/pipelines and serving them for inference.

## joblib (recommended for sklearn objects)

In [ ]:
import joblib
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)

joblib.dump(model, "model.joblib")
loaded_model = joblib.load("model.joblib")
loaded_model.predict(X_test)


## Saving a Full Pipeline (preprocessing + model together)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(random_state=42)),
]).fit(X_train, y_train)

joblib.dump(pipe, "pipeline.joblib")
loaded_pipe = joblib.load("pipeline.joblib")
loaded_pipe.predict(X_test_raw)   # takes raw, unscaled input — pipeline handles preprocessing internally


## pickle (standard library alternative)

In [ ]:
import pickle

with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("model.pkl", "rb") as f:
    loaded = pickle.load(f)


## Compression (smaller files on disk)

In [ ]:
joblib.dump(model, "model_compressed.joblib", compress=3)   # 0 (none) to 9 (max)


## Versioning & Reproducibility Concerns

In [ ]:
import sklearn
print(sklearn.__version__)   # record this alongside the saved model

# store metadata alongside the model for later reference
metadata = {
    "sklearn_version": sklearn.__version__,
    "feature_names": list(X_train.columns),
    "training_date": "2026-07-10",
    "metrics": {"accuracy": 0.94, "f1_macro": 0.92},
}
joblib.dump({"model": pipe, "metadata": metadata}, "model_with_metadata.joblib")

bundle = joblib.load("model_with_metadata.joblib")
bundle["model"].predict(X_test_raw)
bundle["metadata"]


## Wrapping for Inference (a minimal serving function)

In [ ]:
def predict_from_dict(record: dict, pipeline) -> dict:
    df = pd.DataFrame([record])
    pred = pipeline.predict(df)[0]
    proba = pipeline.predict_proba(df)[0].max()
    return {"prediction": str(pred), "confidence": float(proba)}

predict_from_dict({"age": 35, "income": 50000, "city": "NY"}, loaded_pipe)


## Minimal Flask/FastAPI-style Serving Sketch

In [ ]:
# example only — illustrates the typical shape of a serving endpoint
# from fastapi import FastAPI
# app = FastAPI()
# model = joblib.load("pipeline.joblib")
#
# @app.post("/predict")
# def predict(payload: dict):
#     df = pd.DataFrame([payload])
#     return {"prediction": model.predict(df).tolist()}


## ONNX Export (cross-platform, language-agnostic inference)

In [ ]:
# pip install skl2onnx onnxruntime
from skl2onnx import to_onnx
import onnxruntime as ort

onnx_model = to_onnx(pipe, X_train.to_numpy().astype(np.float32))
with open("pipeline.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())

session = ort.InferenceSession("pipeline.onnx")
input_name = session.get_inputs()[0].name
preds = session.run(None, {input_name: X_test.to_numpy().astype(np.float32)})


## Common Gotchas
- Loading a pickled/joblib model requires the SAME (or compatible) scikit-learn version used to save it — mismatches can silently break or raise errors.
- Never unpickle a model from an untrusted source — pickle/joblib execute arbitrary code on load.
- Save the whole `Pipeline`, not just the final estimator, so preprocessing steps travel with the model into production.
- Store feature names/order alongside the model — a plain `.predict(df)` call silently breaks if column order changes for models that don't validate feature names.